In [1]:
import os
import sys
from datasets import load_dataset, Dataset
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
load_dotenv()

d:\chip\project\movie-review-sentiment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU model: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU model: NVIDIA GeForce RTX 3060 Laptop GPU


## LOAD DATA FROM MINIO

In [ ]:
# endpoint_url = os.getenv("ENDPOINT_URL")
# key = os.getenv("MINIO_ROOT_USER")
# secret = os.getenv("MINIO_ROOT_PASSWORD")

In [ ]:
# def load_data_for_nlp(s3a_path):
#     print(f"[*] Đang kết nối tới MinIO tại {endpoint_url}...")
#     print(f"[*] Nạp dữ liệu từ: {s3a_path}")
#     try:
#         dataset = load_dataset(
#             "parquet",
#             data_files=s3a_path,
#             storage_options={
#                 "key": key,
#                 "secret": secret,
#                 "client_kwargs":{
#                     "endpoint_url": endpoint_url
#                 }
#             }
#         )
#         hf_dataset = dataset["train"]
#         print(f"Đã nạp xong Dataset!")
#         print(f"Tổng số bản ghi {len(hf_dataset)}")
#         print(f"Các cột dữ liệu: {hf_dataset.column_names}")
        
#         return hf_dataset
#     except Exception as e:
#         print(f"[!] Lỗi khi nạp dữ liệu trực tiếp: {e}")
#         return None

In [ ]:
# # path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))
# # print(path)
# bucket_name = "silver"
# base_prefix = "nlp/dataset"
# s3_path = get_layer_path("s3://", bucket_name, base_prefix, "2026/03/30") 
# print(s3_path)
# dataset = load_data_for_nlp(s3_path + "*.parquet")
# if dataset:
#     # Kiểm tra thử 1 dòng để xem cấu trúc
#     print("\n[Mẫu dữ liệu dòng 0]:")
#     print(dataset[0])



s3://silver/nlp/dataset/2026/03/30/
[*] Đang kết nối tới MinIO tại http://localhost:9000...
[*] Nạp dữ liệu từ: s3://silver/nlp/dataset/2026/03/30/*.parquet
Đã nạp xong Dataset!
Tổng số bản ghi 70078
Các cột dữ liệu: ['review_id', 'imdb_id', 'content', 'rating', 'source_system']

[Mẫu dữ liệu dòng 0]:
{'review_id': '000524abf6de59d91f3e4a4375ec8415cc5a9cdf1fe595fe334662b4f298d362', 'imdb_id': 'tt27543632', 'content': "For the first hour of the movie, I honestly didn't even know why I was there. It felt cringey, overly sweet, and like I was a 12-year-old getting excited over BookTok edits. But then everything changed. The story completely flipped, the tone turned tense, dark, and slightly psychological. And then came the insane plot twist which was simply wow.\n\nLooking back, the overly innocent and simple beginning was clearly intentional, and that's exactly what made the twist hit so hard. It felt like it came out of nowhere, in the best possible way.\n\nGreat casting, strong perform

In [ ]:
# print(type(dataset))
# for i in range(5):
#     for key, values in dataset[i].items():
#         print(f"{key}: {values}")
#     print("\n")

<class 'datasets.arrow_dataset.Dataset'>
review_id: 000524abf6de59d91f3e4a4375ec8415cc5a9cdf1fe595fe334662b4f298d362
imdb_id: tt27543632
content: For the first hour of the movie, I honestly didn't even know why I was there. It felt cringey, overly sweet, and like I was a 12-year-old getting excited over BookTok edits. But then everything changed. The story completely flipped, the tone turned tense, dark, and slightly psychological. And then came the insane plot twist which was simply wow.

Looking back, the overly innocent and simple beginning was clearly intentional, and that's exactly what made the twist hit so hard. It felt like it came out of nowhere, in the best possible way.

Great casting, strong performances, beautiful cinematography, and a solid soundtrack. The writing and story are sharp and well-crafted.

It's only the beginning of the year, but I already believe this might be the movie of 2026.
rating: 7.0
source_system: imdb


review_id: 000a465558f22ae39464a9830ce91dbcd77

In [3]:
df = pd.read_csv("output.csv")
print(len(df))
df_cleaned = df.dropna(subset=['rating', 'content'])
print(f"[*] Số lượng bản ghi còn lại sau khi lọc null: {len(df)}")

70078
[*] Số lượng bản ghi còn lại sau khi lọc null: 70078


In [4]:
dataset = Dataset.from_pandas(df_cleaned, preserve_index=False)

## PREPROCESSING DATA

In [5]:
dataset = dataset.filter(lambda x: x['rating'] is not None and x['content'] is not None)
print(f"[*] Số lượng bản ghi còn lại sau khi lọc null: {len(dataset)}")

Filter: 100%|██████████| 67803/67803 [00:00<00:00, 153951.55 examples/s]

[*] Số lượng bản ghi còn lại sau khi lọc null: 67803


In [6]:
# Cell: Preprocessing
def clean_text(text):
    import re
    # Ép kiểu string để tránh lỗi "float object has no attribute lower" khi đọc từ CSV
    text = str(text).lower() 
    text = re.sub(r'<.*?>', '', text) # Xóa HTML
    text = re.sub(r'[^a-z0-9.,!?\s]', '', text) # Giữ dấu câu quan trọng
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_fn(examples):
    # Xử lý text
    examples["content_clean"] = [clean_text(t) for t in examples["content"]]
    
    # Scale rating 1-10 về 0-1
    labels = []
    for r in examples["rating"]:
        try:
            val = (float(r) - 1.0) / 9.0
            labels.append(val)
        except (TypeError, ValueError):
            labels.append(0.5) 
            
    examples["label"] = labels
    return examples

# Áp dụng map trên dataset đã được chuyển đổi từ Pandas
dataset_cleaned = dataset.map(preprocess_fn, batched=True)

Map: 100%|██████████| 67803/67803 [00:05<00:00, 12760.59 examples/s]


In [7]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
from datasets import Dataset, DatasetDict

# 1. Chuyển Dataset sang Pandas để xử lý index theo Group
df = dataset_cleaned.to_pandas()

# 2. Khởi tạo bộ chia theo review_id (80% Train, 20% còn lại cho Val & Test)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['review_id']))

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

# 3. Chia 20% còn lại thành Val (10%) và Test (10%)
gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['review_id']))

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

# 4. Đưa ngược về Hugging Face DatasetDict để tiện Tokenize
final_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

print(f"Kết quả chia tập dữ liệu:")
print(f"- Train: {len(final_dataset['train'])} dòng")
print(f"- Val:   {len(final_dataset['validation'])} dòng")
print(f"- Test:  {len(final_dataset['test'])} dòng")

Kết quả chia tập dữ liệu:
- Train: 54242 dòng
- Val:   6780 dòng
- Test:  6781 dòng


In [8]:
from transformers import AutoTokenizer

# Load tokenizer của BERT
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # Truncation: Cắt câu nếu quá dài (>512 tokens)
    # Padding: Thêm 0 nếu câu quá ngắn
    return tokenizer(
        examples["content_clean"], 
        truncation=True, 
        padding="max_length", 
        max_length=256 # Chip có thể tăng lên 512 nếu RAM server .125 còn trống nhiều
    )

# Chạy Tokenize song song trên cả 3 tập
print("[*] Đang Tokenize dữ liệu...")
tokenized_datasets = final_dataset.map(tokenize_function, batched=True)

# Xóa các cột không cần thiết để giải phóng RAM trước khi train
tokenized_datasets = tokenized_datasets.remove_columns(["content", "content_clean", "review_id", "rating"])
tokenized_datasets.set_format("torch")

print("✅ Đã chuẩn bị xong dữ liệu số hóa!")
print(f"Mẫu dữ liệu nạp vào BERT: {tokenized_datasets['train'][0].keys()}")

[*] Đang Tokenize dữ liệu...


Map: 100%|██████████| 6781/6781 [00:01<00:00, 5250.81 examples/s]

✅ Đã chuẩn bị xong dữ liệu số hóa!
Mẫu dữ liệu nạp vào BERT: dict_keys(['imdb_id', 'source_system', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])


In [9]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # BERT Regression trả về mảng (batch_size, 1), cần đưa về mảng 1 chiều
    predictions = predictions.squeeze()
    
    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = np.sqrt(mse)
    
    # Tính toán sai số thực tế trên thang điểm 10 để Chip dễ quan sát
    mae_scale_10 = mae * 9.0
    
    return {
        "mae": mae,
        "rmse": rmse,
        "mae_on_scale_10": mae_scale_10
    }

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load model với đầu ra là 1 node duy nhất (Regression)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=1)
training_args = TrainingArguments(
    output_dir="../models/results_bert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    fp16=True,
    dataloader_num_workers=4
)

# 3. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5383.10it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse,Mae On Scale 10
1,0.025862,0.023801,0.110010,0.154277,0.990087
2,0.017658,0.021936,0.101783,0.148106,0.916051
3,0.011807,0.021387,0.099993,0.146244,0.899934


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.72it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=10173, training_loss=0.021068223350275164, metrics={'train_runtime': 2211.189, 'train_samples_per_second': 73.592, 'train_steps_per_second': 4.601, 'total_flos': 2.1407312587908096e+16, 'train_loss': 0.021068223350275164, 'epoch': 3.0})

In [ ]:
print("[*] Đang đánh giá BERT trên tập Test...")
roberta_test_results = roberta_trainer.predict(roberta_tokenized_datasets["test"])

print("\nKẾT QUẢ CỦA BERT:")
print(f"- MAE (0-1): {roberta_test_results.metrics['test_mae']:.4f}")
print(f"- RMSE (0-1): {roberta_test_results.metrics['test_rmse']:.4f}")
print(f"- Sai số trung bình: {roberta_test_results.metrics['test_mae_on_scale_10']:.2f} điểm")

In [12]:
device = torch.device("cuda")
def predict_rating(text):
    # 1. Làm sạch văn bản (sử dụng lại hàm clean_text bạn đã định nghĩa ở trên)
    text_cleaned = clean_text(text)
    
    # 2. Số hóa (Tokenize) và đưa dữ liệu lên GPU
    inputs = tokenizer(text_cleaned, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    
    # 3. Đảm bảo model đang ở chế độ đánh giá và nằm trên GPU
    model.eval()
    model.to(device)
    
    # 4. Thực hiện dự báo (không tính đạo hàm để tiết kiệm RAM)
    with torch.no_grad():
        outputs = model(**inputs)
        
    # 5. Lấy giá trị đầu ra (đang ở scale 0-1) và kéo giãn về lại thang 1-10
    prediction_0_1 = outputs.logits.item()
    rating_10 = (prediction_0_1 * 9.0) + 1.0
    
    # Đảm bảo điểm số không bị vọt ra ngoài giới hạn [1.0, 10.0]
    return max(1.0, min(10.0, rating_10))

In [15]:
# Chạy thử với những câu mang sắc thái rõ ràng
sample_review = """This movie is a spell binding experience. It is an almost perfect amalgamation of the various departments of film making starting from screenplay, dialogue, direction, sets, acting, cinematography, editing, music, sound, makeup and costume. With 2 hours 45 minute runtime the movie starts in a slightly sluggish manner but halfway through it picks up momentum and the last 45 minutes is superbly presented high drama. The setting is a desert area of sand dunes, with shifting sands and sand storms unfolding before you as an awe inspiring spectacle.
There is intrigue and action with people with mystic masks performing bizarre rituals. The sound effects of whirring machines, vehicles moving through sand and explosions are stupendous. Scenes of the Gladiator arena have fabulous visuals. There are flying aircraft of unusual designs shown carrying out attacks. All with mesmerizing BGM.
Take a bow director Denis Villeneuve for creating this marvel ( pun intended) of a movie of stupendous proportions and for extacting the best from writers, actors and technicians. He has made the scenes totally believeable even though some costumes and make up appear bizarre. Acting wise Timothee, Zendaya and Javier have prominent roles and they have given superb performances. Music by Hans Zimmer is top class. This movie could well be a coaching class for cinematographers , editors and sound recordists. The connoisseurs among the audience in any case will relish each scene."""
sample_review2 = """Best movie in the trilogy and sealed in the best possible way"""
sample_review3 = """Some films are so bad, they're good. Meaning "fun" or "entertaining" or at least "interesting"... But few films can claim to be so dreadful, you actually feel physical pain while viewing them. So bad are these damned few that you don't experience them or watch them: you "endure" them. This is the mother of all such films!

"Manos the hands of fate" is without a doubt the most inept and atrociously awful film ever made. Its poorness is so extreme that of itself it is the film's strongest selling point. The script is non-existent, the acting makes Steven Seagal look like a member of the Royal Shakespeare Company and the editing could have been less horrendously botched by a blind Eskimo with no arms. It is also painfully slow. This film makes might barely last 70 minutes but you will feel like you've aged ten years by the end of it. That's what makes a film truly bad: the fact that despite its overbearing weaknesses it isn't even entertaining!

Many people look back at the sixties and think, with obvious resentment for today's cinematic output, that "they don't make them like this anymore!". Watching "Manos..." would cure any breed of hardcore nostalgia.

In the end I can not advise against this strongly enough. This is for the masochist in you (or the sadist if you insist on showing it to friends). Any other part of your person can only feel pained or offended by such extreme trash!"""
test_reviews = [sample_review3]
for review in test_reviews:
    score = predict_rating(review)
    print(f"Review: '{review}'")
    print(f"AI chấm: {score:.1f}/10\n{'-'*40}")

Review: 'Some films are so bad, they're good. Meaning "fun" or "entertaining" or at least "interesting"... But few films can claim to be so dreadful, you actually feel physical pain while viewing them. So bad are these damned few that you don't experience them or watch them: you "endure" them. This is the mother of all such films!

"Manos the hands of fate" is without a doubt the most inept and atrociously awful film ever made. Its poorness is so extreme that of itself it is the film's strongest selling point. The script is non-existent, the acting makes Steven Seagal look like a member of the Royal Shakespeare Company and the editing could have been less horrendously botched by a blind Eskimo with no arms. It is also painfully slow. This film makes might barely last 70 minutes but you will feel like you've aged ten years by the end of it. That's what makes a film truly bad: the fact that despite its overbearing weaknesses it isn't even entertaining!

Many people look back at the sixti

In [ ]:
# Cell: Đánh giá trên tập Test
print("[*] Đang đánh giá mô hình trên tập Test...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n📊 KẾT QUẢ CUỐI CÙNG:")
print(f"- MAE (0-1): {test_results['eval_mae']:.4f}")
print(f"- RMSE (0-1): {test_results['eval_rmse']:.4f}")
print(f"- Sai số trung bình (thang điểm 10): {test_results['eval_mae_on_scale_10']:.2f} điểm")

In [ ]:
# Cell: Lưu mô hình
model_path = "./models/bert-movie-rating-regressor"

# Lưu trọng số mô hình
trainer.save_model(model_path)
# Lưu bộ giải mã (tokenizer) - Rất quan trọng để tái cấu trúc văn bản sau này
tokenizer.save_pretrained(model_path)

print(f"✅ Đã lưu mô hình tại: {model_path}")